# Sacred-Texts Rigveda Translator & Modernizer

This notebook fetches Rigveda hymns from [sacred-texts.com](https://sacred-texts.com/hin/rigveda/index.htm), rewrites them from archaic English into modern, natural English with Hinglish/Sanskrit terminology (e.g., *Surya*, *Agni*, *Indra*, *Yajna*, *Devas*), and exports a consolidated text file.

### Google Colab Instructions:
1. Go to **Runtime** > **Change runtime type** in the top menu.
2. Select **T4 GPU** (or any available GPU) and click **Save** (this accelerates LLM inference significantly).
3. Run all the cells in order.

In [ ]:
# Install the necessary libraries for scraper and LLM
!pip install -q curl-cffi beautifulsoup4 transformers accelerate bitsandbytes torch

In [ ]:
import time
import urllib.parse
from typing import Any, Dict, List, Optional
from bs4 import BeautifulSoup
from curl_cffi import requests

class SacredTextsScraper:
    def __init__(
        self,
        index_url: str = "https://sacred-texts.com/hin/rigveda/index.htm",
        delay: float = 1.0,
        max_retries: int = 3,
        book_selector: str = "body > a[href]",
        hymn_selector: str = "body > a[href]",
        hymn_content_selector: str = "body > p",
        hymn_p_index: Optional[int] = None
    ):
        self.index_url = index_url
        self.delay = delay
        self.max_retries = max_retries
        self.book_selector = book_selector
        self.hymn_selector = hymn_selector
        self.hymn_content_selector = hymn_content_selector
        self.hymn_p_index = hymn_p_index
        
        parsed_url = urllib.parse.urlparse(self.index_url)
        self.base_url = f"{parsed_url.scheme}://{parsed_url.netloc}{parsed_url.path.rsplit('/', 1)[0]}/"
        
    def _fetch_html(self, url: str) -> Optional[str]:
        """Fetch HTML content using curl_cffi to bypass Cloudflare."""
        retries = 0
        backoff = 2.0
        
        while retries <= self.max_retries:
            try:
                response = requests.get(
                    url, 
                    impersonate="chrome", 
                    timeout=30,
                    allow_redirects=True
                )
                if response.status_code == 200:
                    if self.delay > 0:
                        time.sleep(self.delay)
                    return response.text
                print(f"[Warning] Non-200 status {response.status_code} for {url}. Retrying...")
            except Exception as e:
                print(f"[Warning] Error fetching {url}: {e}. Retrying...")
            
            time.sleep(backoff)
            retries += 1
            backoff *= 2.0
            
        print(f"[Error] Failed to fetch {url} after {self.max_retries} attempts.")
        return None

    def _extract_text_heuristic(self, p_tags: List[BeautifulSoup]) -> str:
        """Extract the actual hymn content from paragraph tags using heuristics."""
        best_p = None
        best_score = -9999.0
        
        for i, p in enumerate(p_tags):
            text = p.get_text().strip()
            if not text:
                continue
                
            links = p.find_all("a")
            text_len = len(text)
            link_density = len(links) / text_len if text_len > 0 else 0
            
            score = -100.0 * link_density
            
            if text_len < 50:
                score -= 10.0
            else:
                score += min(text_len / 100.0, 15.0)
                
            keywords = ["next:", "previous:", "index", "sacred texts", "sacred-texts", "buy this book", "translated by", "tr. by"]
            lower_text = text.lower()
            keyword_count = sum(1 for kw in keywords if kw in lower_text)
            
            if keyword_count > 0:
                score -= 50.0 * keyword_count
                
            words = text.split()
            if words and words[0].isdigit():
                score += 20.0
                
            if score > best_score:
                best_score = score
                best_p = p
                
        if best_p:
            for br in best_p.find_all("br"):
                br.replace_with("\n")
            return best_p.get_text().strip()
            
        return ""

    def get_books_list(self) -> List[Dict[str, str]]:
        """Retrieve sub-books/mandalas list from index."""
        html = self._fetch_html(self.index_url)
        if not html:
            raise RuntimeError("Failed to retrieve index URL")
        soup = BeautifulSoup(html, "html.parser")
        book_elements = soup.select(self.book_selector)
        
        books = []
        seen_urls = set()
        
        for elem in book_elements:
            href = elem.get("href")
            if not href:
                continue
            absolute_url = urllib.parse.urljoin(self.index_url, href)
            if absolute_url.startswith(self.base_url) and absolute_url != self.index_url:
                if absolute_url not in seen_urls and "errata.htm" not in absolute_url:
                    books.append({
                        "title": elem.get_text().strip(),
                        "url": absolute_url
                    })
                    seen_urls.add(absolute_url)
        return books

    def get_hymns_list(self, book_url: str) -> List[Dict[str, str]]:
        """Retrieve hymns list from a sub-book/mandala page."""
        html = self._fetch_html(book_url)
        if not html:
            return []
        soup = BeautifulSoup(html, "html.parser")
        hymn_elements = soup.select(self.hymn_selector)
        
        hymns = []
        seen_urls = set()
        
        for elem in hymn_elements:
            href = elem.get("href")
            if not href or "index.htm" in href or "errata.htm" in href or "../" in href:
                continue
            absolute_url = urllib.parse.urljoin(book_url, href)
            if absolute_url.startswith(self.base_url) and absolute_url not in seen_urls:
                hymns.append({
                    "title": elem.get_text().strip(),
                    "url": absolute_url
                })
                seen_urls.add(absolute_url)
        return hymns

    def scrape_hymn_content(self, hymn_url: str) -> str:
        """Fetch a single hymn content and extract the verses."""
        html = self._fetch_html(hymn_url)
        if not html:
            return ""
        soup = BeautifulSoup(html, "html.parser")
        p_tags = soup.select(self.hymn_content_selector)
        
        if self.hymn_p_index is not None and 0 <= self.hymn_p_index < len(p_tags):
            target_p = p_tags[self.hymn_p_index]
            for br in target_p.find_all("br"):
                br.replace_with("\n")
            return target_p.get_text().strip()
            
        return self._extract_text_heuristic(p_tags)
print("Scraper engine loaded successfully!")

## Setup Local LLM (`Qwen/Qwen2.5-3B-Instruct`)

We load the model in 4-bit precision to reduce the memory footprint. This allows it to run smoothly on Colab's standard T4 GPU instance.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Selected execution device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)

if device == "cuda":
    print("GPU detected. Loading model in 4-bit precision...")
    # Configure 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    print("GPU not detected. Loading model in standard precision on CPU (this will be slower)...")
    # Fallback to float32 on CPU
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="auto"
    )

print("Model loaded successfully!")

## Rigveda Modernizer Function

We set up a prompt template that directs the model to perform translation into modern English while keeping Sanskrit/Hinglish terms for key religious and spiritual concepts.

In [ ]:
system_prompt = (
    "You are an expert translator and Vedic scholar. Your task is to rewrite the given Rigvedic hymn from archaic/old English into clear, modern English.\n"
    "Follow these guidelines:\n"
    "1. Maintain the original meaning and spiritual essence. Do not add external interpretations.\n"
    "2. Use modern, natural English phrasing. Avoid archaic words like 'laud', 'thou', 'thy', 'hitherward', 'hotar', 'oblation', etc.\n"
    "3. Use standard Sanskrit/Hinglish terms for deities and key concepts instead of their English translations where appropriate. For example:\n"
    "   - Use 'Surya' instead of 'Sun' or 'sun god'.\n"
    "   - Use 'Agni' instead of 'Fire' or 'fire god'.\n"
    "   - Use 'Indra' instead of 'thunder god' or 'rain god'.\n"
    "   - Use 'Soma' for the sacred beverage.\n"
    "   - Use 'Yajna' or 'Havan' instead of 'sacrifice' or 'offering'.\n"
    "   - Use 'Deva' or 'Devas' instead of 'God' or 'Gods'.\n"
    "   - Use 'Mantra' or 'Shloka' instead of 'prayer' or 'song' or 'hymn'.\n"
    "   - Use 'Rishi' instead of 'seer' or 'priest'.\n"
    "4. Output ONLY the rewritten modern English translation of the verses. Do not include introductory notes, explanations, or chatty sentences."
)

def modernize_hymn(text: str) -> str:
    if not text.strip():
        return ""
        
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Original Hymn Content:\n{text}"}
    ]
    
    # Form prompt using model-specific chat template
    text_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=1024,
            temperature=0.3,
            do_sample=True
        )
        
    # Trim prompt input from output ids
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.strip()

# Quick sanity check to verify translation
sample_text = "1 I Laud Agni, the chosen Priest, God, minister of sacrifice, The hotar, lavishest of wealth."
print("Sample translation check:")
print(f"Original: {sample_text}")
print(f"Modernized: {modernize_hymn(sample_text)}")

## Start Scraping and Translation Loop

We initialize the scraper and run the loop book-by-book. You can set `LIMIT_HYMNS = None` to process all hymns, or specify an integer limit for a quick test run.

In [ ]:
# --- Configuration ---
LIMIT_HYMNS = None     # Set to None to scrape all hymns across all mandalas
DELAY_SECONDS = 0.5  # Politeness delay between requests
# ---------------------

scraper = SacredTextsScraper(delay=DELAY_SECONDS)

print("Fetching books/mandalas list...")
books = scraper.get_books_list()
print(f"Found {len(books)} books/mandalas.\n")

hymns_processed = 0
output_lines = [
    "RIG VEDA - MODERN ENGLISH & HINGLISH REWRITE",
    "=" * 50,
    ""
]

limit_reached = False

for book_idx, book in enumerate(books):
    if limit_reached:
        break
        
    print(f"Processing Book {book_idx + 1}/{len(books)}: '{book['title']}'")
    output_lines.append("=" * 50)
    output_lines.append(book['title'].upper())
    output_lines.append("=" * 50)
    output_lines.append("")
    
    hymns = scraper.get_hymns_list(book['url'])
    print(f"Found {len(hymns)} hymns in this book. Processing...")
    
    for hymn_idx, hymn in enumerate(hymns):
        if LIMIT_HYMNS is not None and hymns_processed >= LIMIT_HYMNS:
            print(f"Reached limit of {LIMIT_HYMNS} hymns. Stopping.")
            limit_reached = True
            break
            
        # 1. Scrape content
        original_content = scraper.scrape_hymn_content(hymn['url'])
        
        # 2. Modernize/Rewrite
        rewritten_content = ""
        if original_content:
            try:
                rewritten_content = modernize_hymn(original_content)
            except Exception as e:
                print(f"Error modernizing hymn {hymn['title']}: {e}")
                rewritten_content = f"[Modernization Error]: {original_content}"
        else:
            print(f"[Warning] Could not extract content for {hymn['title']}")
            
        # 3. Format and save
        output_lines.append("-" * 50)
        output_lines.append(f"{hymn['title']}")
        output_lines.append(f"URL: {hymn['url']}")
        output_lines.append("-" * 50)
        output_lines.append(rewritten_content)
        output_lines.append("")
        output_lines.append("")
        
        hymns_processed += 1
        if hymns_processed % 5 == 0 or LIMIT_HYMNS is not None:
            print(f"  Processed {hymns_processed} hymns...")
            
print(f"\nProcessing complete! Total hymns modernized: {hymns_processed}")

## Save and Download Consolidated File

This cell writes all rewritten hymns into a single formatted text file and starts the download in your browser.

In [ ]:
output_filename = "rigveda_modern.txt"

# Write file content
with open(output_filename, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))
    
print(f"Saved consolidated content to {output_filename}.")

# Trigger browser download inside Google Colab
try:
    from google.colab import files
    print("Starting browser download...")
    files.download(output_filename)
except ImportError:
    print("[Notice] google.colab is not available. Download link skipped.")
    print(f"You can find the file locally at: {output_filename}")